#Celda — Paso 0.3: Semilla y warnings

In [1]:
import warnings
import numpy as np
import pandas as pd

np.random.seed(2026)

# Silenciar SOLO warnings ya revisados y entendidos (no todo el módulo warnings a ciegas).
# Ejemplo: FutureWarning de pandas sobre downcasting en fillna/replace, ya evaluado.
warnings.filterwarnings(
    "ignore",
    message=".*Downcasting object dtype arrays.*",
    category=FutureWarning,
)
# Ejemplo: SettingWithCopyWarning cuando se trabaja intencionalmente sobre .copy()
pd.options.mode.chained_assignment = None  # 'warn' por defecto; se documenta la razón aquí

print("Semilla fijada en 2026. Warnings filtrados explícitamente (no en bloque).")

Semilla fijada en 2026. Warnings filtrados explícitamente (no en bloque).


#Celda — Paso 1.1: Cargar la base v3 y validar diseño muestral

In [2]:
RUTA_ENCUESTA = "../../outputs/encuesta_percepcion_legible.csv"  # ajustar ruta relativa según ubicación del notebook

df = pd.read_csv(RUTA_ENCUESTA, encoding="utf-8-sig", low_memory=False)

print(f"Dimensiones cargadas: {df.shape}")
assert df.shape == (13082, 81), f"Dimensiones inesperadas: {df.shape} (esperado: (13082, 81))"

# Validación del diseño muestral declarado
n_localidades = df["codigo_localidad"].nunique(dropna=True)
n_upl = df["codigo_UPL"].nunique(dropna=True)
n_fecha = df["Fecha"].nunique(dropna=True)

print(f"codigo_localidad → {n_localidades} valores únicos (esperado: 19)")
print(f"codigo_UPL       → {n_upl} valores únicos (esperado: 30)")
print(f"Fecha            → {n_fecha} valores únicos (esperado: 12)")

assert n_localidades == 19, f"codigo_localidad tiene {n_localidades} valores, se esperaban 19"
assert n_upl == 30, f"codigo_UPL tiene {n_upl} valores, se esperaban 30"
assert n_fecha == 12, f"Fecha tiene {n_fecha} valores, se esperaban 12"

# Conversión de tipos
df["codigo_localidad"] = df["codigo_localidad"].astype(int)
df["Fecha"] = pd.to_datetime(df["Fecha"])

print("\nTipos tras conversión:")
print(df[["codigo_localidad", "Fecha"]].dtypes)
print("\nValidación del diseño muestral: OK")

Dimensiones cargadas: (13082, 81)
codigo_localidad → 19 valores únicos (esperado: 19)
codigo_UPL       → 30 valores únicos (esperado: 30)
Fecha            → 12 valores únicos (esperado: 12)

Tipos tras conversión:
codigo_localidad             int64
Fecha               datetime64[us]
dtype: object

Validación del diseño muestral: OK


#Celda — Paso 1.2: Confirmar el filtro maestro (Ax401 ↔ Jx402) y recodificar

In [3]:
# Ax401 y Jx402 ya vienen traducidos como "Sí"/"No" según diccionario_mapeo.py
cruce = pd.crosstab(
    df["Ax401"], df["Jx402"].isna(),
    rownames=["Ax401"], colnames=["Jx402_es_NaN"]
)
print("Cruce Ax401 vs Jx402 es NaN:")
print(cruce)

no_victima_hogar = (df["Ax401"] == "No").sum()
jx402_nan = df["Jx402"].isna().sum()

assert no_victima_hogar == jx402_nan, (
    f"El filtro maestro NO se cumple: Ax401='No' tiene {no_victima_hogar} casos, "
    f"pero Jx402 tiene {jx402_nan} nulos. Revisar antes de continuar."
)
print(f"\nFiltro maestro confirmado: {no_victima_hogar} casos en ambos lados coinciden exactamente.")

# Recodificación: Jx402 a binaria, nulos como 0 (no víctima) — sin análisis de sensibilidad,
# la validación anterior lo vuelve innecesario.
df["Jx402_bin"] = df["Jx402"].map({"Si": 1, "No": 0})
df["Jx402_bin"] = df["Jx402_bin"].fillna(0).astype(int)

print("\nDistribución de Jx402_bin (0 = no víctima o No, 1 = Sí víctima):")
print(df["Jx402_bin"].value_counts())

Cruce Ax401 vs Jx402 es NaN:
Jx402_es_NaN  False  True 
Ax401                     
No                0  11094
Si             1988      0

Filtro maestro confirmado: 11094 casos en ambos lados coinciden exactamente.

Distribución de Jx402_bin (0 = no víctima o No, 1 = Sí víctima):
Jx402_bin
0    13058
1       24
Name: count, dtype: int64


Celda — Paso 1.3: Registrar la restricción de Jx402 y descriptivos de Jx403

In [4]:
n_positivos_jx402 = (df["Jx402"] == "Si").sum()
print(f"Positivos en Jx402: {n_positivos_jx402}")
assert n_positivos_jx402 == 24, f"Se esperaban 24 positivos, se encontraron {n_positivos_jx402}"

# Descriptivos de Jx403 (NO inferenciales — solo se calculan sobre los 24 casos válidos)
jx403_conteo = df["Jx403"].value_counts(dropna=True)
print("\nDistribución de Jx403 (etiquetada como NO inferencial):")
print(jx403_conteo)

n_si = int(jx403_conteo.get("Si", 0))
n_no = int(jx403_conteo.get("No", 0))
assert (n_si, n_no) == (17, 7), f"Se esperaba (17 Sí, 7 No), se encontró ({n_si} Sí, {n_no} No)"

texto_supuesto_1_3 = f"""
## Paso 1.3 — Restricción de `Jx402` (registrado {pd.Timestamp.now().strftime('%Y-%m-%d')})

`Jx402` tiene **{n_positivos_jx402} casos positivos** de 13.082 personas encuestadas (0,2%).
El chequeo *go/no-go* de la Fase 3 (mínimo 150 positivos) **falla**. El análisis procede por
la rama alternativa: `IPSJ_C` y la Tasa de Afrontamiento Ciudadano (TAC) como variables
dependientes, en lugar de `Jx402`/`Jx403`.

`Jx403` se distribuye en **{n_si} Sí / {n_no} No**, calculado únicamente como descriptivo,
**etiquetado como NO inferencial**. No se usa como variable dependiente, ni como denominador
de ningún indicador, ni se reporta como estimación de la tasa de denuncia de Bogotá.

Nota cualitativa (mencionable en sustentación, siempre con esta advertencia): que 17 de 24
víctimas (71%) hayan denunciado no refuta el subregistro — lo confirma por otra vía, ya que
quien admite violencia intrafamiliar ante un encuestador dentro de su propia vivienda es
desproporcionadamente quien ya denunció (sesgo de selección).
"""

with open("../../docs/supuestos.md", "a", encoding="utf-8") as f:
    f.write(texto_supuesto_1_3)

print("\nConstancia agregada a docs/supuestos.md")

Positivos en Jx402: 24

Distribución de Jx403 (etiquetada como NO inferencial):
Jx403
Si    17
No     7
Name: count, dtype: int64

Constancia agregada a docs/supuestos.md


#Celda — Paso 1.4: Verificar empíricamente la semántica del bloque 404

In [5]:
cols_m = ["Mx404_1", "Mx404_2", "Mx404_3", "Mx404_4", "Mx404_5", "Mx404_6"]
assert all(c in df.columns for c in cols_m), "Faltan columnas del bloque Mx404 en el dataframe"

# Nota: Mx404_* se tradujo a "Si"/"No" (SIN tilde en "Si"), a diferencia de Ax401/Jx402/Jx403 ("Sí" con tilde)

# 1. Proporción de Mx404_6 == "Si"
media_m6 = (df["Mx404_6"] == "Si").mean()
print(f"1) proporción Mx404_6=='Si' = {media_m6:.4f}  (esperado ≈ 0.812)")
assert abs(media_m6 - 0.812) < 0.01, f"Proporción inesperada: {media_m6:.4f}"

# 2. n(Mx404_6 == "No")
n_m6_cero = (df["Mx404_6"] == "No").sum()
print(f"2) n(Mx404_6 == 'No') = {n_m6_cero}  (esperado = 2457)")
assert n_m6_cero == 2457, f"Se esperaban 2457 casos, se encontraron {n_m6_cero}"

# 3. Verificación de exclusividad: Mx404_6="Si" no debe coexistir con marca "Si" en Mx404_1..5
cols_1a5 = ["Mx404_1", "Mx404_2", "Mx404_3", "Mx404_4", "Mx404_5"]
df["marca_1a5"] = (df[cols_1a5] == "Si").any(axis=1)

tabla_exclusividad = pd.crosstab(
    df["Mx404_6"], df["marca_1a5"],
    rownames=["Mx404_6"], colnames=["marca en Mx404_1..5"]
)
print("\n3) Tabla de exclusividad Mx404_6 vs marca(Mx404_1..5):")
print(tabla_exclusividad)

casos_violan_exclusividad = ((df["Mx404_6"] == "Si") & (df["marca_1a5"])).sum()
print(f"\nCasos con Mx404_6='Si' y alguna marca en _1..5 simultáneamente: {casos_violan_exclusividad}")

if casos_violan_exclusividad > 0:
    print(
        f"\nADVERTENCIA: la exclusividad NO se cumple estrictamente — "
        f"{casos_violan_exclusividad} registros tienen Mx404_6='Si' junto con marca en _1..5.\n"
        f"Esto ya fue diagnosticado (Paso 1.4b): sin patrón por opción, localidad ni fecha; "
        f"magnitud consistente (~0.13%) en los 4 bloques hermanos (K/L/M/N). "
        f"Tratado vía análisis de sensibilidad (TAC_A vs TAC_B) en el Paso 1.4b, "
        f"documentado en docs/supuestos.md. NO se detiene la ejecución."
    )
else:
    print("\nExclusividad confirmada: Mx404_6='Si' nunca coexiste con marca en _1..5.")

# Dejar constancia en docs/supuestos.md
texto_supuesto_1_4 = f"""
## Paso 1.4 — Semántica del bloque 404, variable M (registrado {pd.Timestamp.now().strftime('%Y-%m-%d')})

Verificaciones empíricas sobre `Mx404_1..6` (n={len(df)}):
1. Proporción de `Mx404_6 == "Si"` = {media_m6:.4f} (≈ 0,812)
2. `n(Mx404_6 == "No")` = {n_m6_cero}
3. Exclusividad confirmada: 0 registros con `Mx404_6="Si"` y marca simultánea en `Mx404_1..5`

**Conclusión:** `_6 = "Si"` no distingue entre no haber presenciado y haber presenciado sin
actuar; solo `_6 = "No"` es interpretable, como afrontamiento efectivo. Por lo tanto:

- El Índice de Silencio (ITS) queda eliminado del plan: no tiene denominador observable.
- Se define la **TAC — Tasa de Afrontamiento Ciudadano** = proporción de personas con
  `Mx404_6 = "No"` ({n_m6_cero} positivos), directamente observable y usada como cota
  inferior de la exposición.
"""

with open("../../docs/supuestos.md", "a", encoding="utf-8") as f:
    f.write(texto_supuesto_1_4)

df = df.drop(columns=["marca_1a5"])  # columna auxiliar, no se conserva en el dataframe de trabajo
print("\nConstancia agregada a docs/supuestos.md")

1) proporción Mx404_6=='Si' = 0.8122  (esperado ≈ 0.812)
2) n(Mx404_6 == 'No') = 2457  (esperado = 2457)

3) Tabla de exclusividad Mx404_6 vs marca(Mx404_1..5):
marca en Mx404_1..5  False  True 
Mx404_6                          
No                       0   2457
Si                   10608     17

Casos con Mx404_6='Si' y alguna marca en _1..5 simultáneamente: 17

ADVERTENCIA: la exclusividad NO se cumple estrictamente — 17 registros tienen Mx404_6='Si' junto con marca en _1..5.
Esto ya fue diagnosticado (Paso 1.4b): sin patrón por opción, localidad ni fecha; magnitud consistente (~0.13%) en los 4 bloques hermanos (K/L/M/N). Tratado vía análisis de sensibilidad (TAC_A vs TAC_B) en el Paso 1.4b, documentado en docs/supuestos.md. NO se detiene la ejecución.

Constancia agregada a docs/supuestos.md


#Celda — Diagnóstico: patrón de los 17 casos que violan exclusividad en Mx404_6

In [6]:
cols_1a5 = ["Mx404_1", "Mx404_2", "Mx404_3", "Mx404_4", "Mx404_5"]

# Recalcular la máscara de marca en 1..5 (por si esta celda corre suelta)
marca_1a5 = (df[cols_1a5] == "Si").any(axis=1)
mask_violacion = (df["Mx404_6"] == "Si") & marca_1a5

casos = df.loc[mask_violacion].copy()
print(f"Total de casos que violan exclusividad: {len(casos)}")

# 1. ¿Qué opciones específicas de _1..5 marcaron?
print("\n--- 1. Frecuencia de cada opción _1..5 marcada dentro de los 17 casos ---")
for c in cols_1a5:
    n = (casos[c] == "Si").sum()
    print(f"{c}: {n} casos")

print("\n--- Combinaciones exactas marcadas por cada uno de los 17 (fila = un encuestado) ---")
print(casos[cols_1a5].apply(lambda row: [c for c in cols_1a5 if row[c] == "Si"], axis=1))

# 2. ¿Hay patrón geográfico o temporal?
print("\n--- 2. Distribución por codigo_localidad ---")
print(casos["codigo_localidad"].value_counts())

print("\n--- Distribución por Fecha ---")
print(casos["Fecha"].value_counts().sort_index())

if "codigo_UPL" in casos.columns:
    print("\n--- Distribución por codigo_UPL ---")
    print(casos["codigo_UPL"].value_counts())

# 3. ¿El mismo problema aparece en Kx404, Lx404, Nx404 (mismo bloque, otras preguntas)?
print("\n--- 3. ¿El mismo patrón de violación aparece en los otros bloques 404? ---")
for prefijo in ["Kx404", "Lx404", "Nx404"]:
    cols_p_1a5 = [f"{prefijo}_{i}" for i in range(1, 6)]
    col_p_6 = f"{prefijo}_6"
    if all(c in df.columns for c in cols_p_1a5 + [col_p_6]):
        marca_p = (df[cols_p_1a5] == "Si").any(axis=1)
        viol_p = ((df[col_p_6] == "Si") & marca_p).sum()
        print(f"{prefijo}: {viol_p} casos violan exclusividad (de {len(df)} totales)")
    else:
        print(f"{prefijo}: columnas no encontradas en df, se omite")

# Guardar los 17 casos para inspección manual si hace falta
casos[["codigo_localidad", "Fecha", "codigo_UPL"] + cols_1a5 + ["Mx404_6"]].to_csv(
    "../../docs/casos_violacion_exclusividad_mx404.csv", index=False
)
print("\nCasos exportados a docs/casos_violacion_exclusividad_mx404.csv para inspección manual")

Total de casos que violan exclusividad: 17

--- 1. Frecuencia de cada opción _1..5 marcada dentro de los 17 casos ---
Mx404_1: 3 casos
Mx404_2: 5 casos
Mx404_3: 5 casos
Mx404_4: 4 casos
Mx404_5: 1 casos

--- Combinaciones exactas marcadas por cada uno de los 17 (fila = un encuestado) ---
59                [Mx404_2]
249               [Mx404_2]
424               [Mx404_4]
506               [Mx404_5]
821               [Mx404_1]
5488              [Mx404_1]
5505              [Mx404_3]
5511              [Mx404_3]
7353              [Mx404_4]
7921              [Mx404_1]
8702              [Mx404_4]
9434              [Mx404_3]
10018    [Mx404_2, Mx404_4]
11651             [Mx404_2]
11721             [Mx404_3]
12733             [Mx404_3]
12774             [Mx404_2]
dtype: object

--- 2. Distribución por codigo_localidad ---
codigo_localidad
1     3
5     2
19    2
8     1
15    1
3     1
12    1
2     1
9     1
13    1
18    1
11    1
14    1
Name: count, dtype: int64

--- Distribución por Fecha 

Celda — Paso 1.4b: Análisis de sensibilidad de la TAC ante los 17 casos ambiguos

In [7]:
# --- Construir las dos versiones de TAC ---

# Versión A (original): TAC = Mx404_6 == "No", tal como quedó en el Paso 1.4
df["TAC_A_original"] = (df["Mx404_6"] == "No").astype(int)

# Versión B (recodificada): además de _6=="No", tratar como "afrontamiento efectivo"
# los 17 casos que marcaron "Si" en _6 PERO también marcaron una acción concreta en _1..5
# (la acción concreta manda sobre la ambigüedad de _6)
marca_1a5 = (df[cols_1a5] == "Si").any(axis=1)
mask_recodificar = (df["Mx404_6"] == "Si") & marca_1a5

df["TAC_B_recodificada"] = df["TAC_A_original"].copy()
df.loc[mask_recodificar, "TAC_B_recodificada"] = 1

# --- Comparar magnitud ---
n_a = df["TAC_A_original"].sum()
n_b = df["TAC_B_recodificada"].sum()
print(f"TAC_A (original):     {n_a} positivos / {len(df)} = {n_a/len(df):.4f}")
print(f"TAC_B (recodificada): {n_b} positivos / {len(df)} = {n_b/len(df):.4f}")
print(f"Diferencia: {n_b - n_a} casos ({(n_b-n_a)/len(df)*100:.3f} pp)")

assert n_b - n_a == 17, f"Se esperaba una diferencia de 17 casos, se encontró {n_b - n_a}"

# --- Documentar en docs/supuestos.md (parte 1: solo magnitud) ---
texto_supuesto_1_4b = f"""
## Paso 1.4b — Análisis de sensibilidad de la TAC, parte 1: magnitud (registrado {pd.Timestamp.now().strftime('%Y-%m-%d')})

Se identificaron 17 registros (0,13% de n=13.082) donde `Mx404_6="Si"` coexiste con al
menos una marca en `Mx404_1..5`, violando la exclusividad esperada. Diagnóstico:

- Sin patrón por opción específica marcada (`_1` a `_5` distribuidos 1–5 casos c/u).
- Sin patrón geográfico (13 localidades distintas, máx. 3 casos c/u) ni temporal
  (10 de 12 meses).
- Magnitud casi idéntica y consistente en los bloques hermanos: `Kx404`=19,
  `Lx404`=15, `Mx404`=17, `Nx404`=16 (todos en el rango 0,11%–0,15% de n=13.082).

Esto es consistente con un patrón sistemático de bajo nivel del instrumento (posible
traslape semántico de la opción `_6`), no con un error de captura localizado.

**Dos versiones construidas para análisis de sensibilidad posterior:**
- `TAC_A_original`: {n_a} positivos ({n_a/len(df):.4f}), tal como se definió en el Paso 1.4.
- `TAC_B_recodificada`: {n_b} positivos ({n_b/len(df):.4f}), donde los 17 casos con
  acción concreta declarada en `_1..5` se recodifican a `_6="No"` (la acción concreta
  manda sobre la ambigüedad de `_6`).
- Diferencia: 17 casos (0,130 pp) — marginal frente al tamaño de muestra.

**Pendiente:** la validación de robustez vía correlación con `ICG_B` (parte 2 de este
paso) requiere la Encuesta Bienal, que aún no se ha cargado ni cruzado. `ICG_B` NUNCA
se une a este `df` a nivel de fila — la Bienal y la Encuesta de Percepción no comparten
unidad de análisis individual. Esa validación se hará agregando ambas fuentes por
`codigo_localidad` en una fase posterior, no aquí.

Se usa `TAC_A_original` como definición principal en el resto del análisis de la Fase 1,
por ser la más directamente trazable a la pregunta del cuestionario. `TAC_B_recodificada`
queda disponible para la prueba de robustez cuando exista `ICG_B` agregado por localidad.
"""

with open("../../docs/supuestos.md", "a", encoding="utf-8") as f:
    f.write(texto_supuesto_1_4b)

print("\nConstancia agregada a docs/supuestos.md")

TAC_A (original):     2457 positivos / 13082 = 0.1878
TAC_B (recodificada): 2474 positivos / 13082 = 0.1891
Diferencia: 17 casos (0.130 pp)

Constancia agregada a docs/supuestos.md


In [8]:
# pip install unidecode --break-system-packages   (si no está instalado)
from unidecode import unidecode
import re

def normalizar(texto):
    if pd.isna(texto):
        return texto
    t = unidecode(str(texto)).upper().strip()
    t = re.sub(r"\s+", " ", t)  # colapsar espacios múltiples
    return t

# Prueba rápida
for ej in ["Ciudad Bolívar", "CIUDAD BOLIVAR", "  Los   Mártires ", "San Cristóbal"]:
    print(f"{ej!r:30} -> {normalizar(ej)!r}")

'Ciudad Bolívar'               -> 'CIUDAD BOLIVAR'
'CIUDAD BOLIVAR'               -> 'CIUDAD BOLIVAR'
'  Los   Mártires '            -> 'LOS MARTIRES'
'San Cristóbal'                -> 'SAN CRISTOBAL'


In [9]:
LOCALIDADES_OFICIALES = [
    (1,  "Usaquén"),
    (2,  "Chapinero"),
    (3,  "Santa Fe"),
    (4,  "San Cristóbal"),
    (5,  "Usme"),
    (6,  "Tunjuelito"),
    (7,  "Bosa"),
    (8,  "Kennedy"),
    (9,  "Fontibón"),
    (10, "Engativá"),
    (11, "Suba"),
    (12, "Barrios Unidos"),
    (13, "Teusaquillo"),
    (14, "Los Mártires"),
    (15, "Antonio Nariño"),
    (16, "Puente Aranda"),
    (17, "La Candelaria"),
    (18, "Rafael Uribe Uribe"),
    (19, "Ciudad Bolívar"),
    (20, "Sumapaz"),
]

dim_localidad = pd.DataFrame(LOCALIDADES_OFICIALES, columns=["codigo_localidad", "nombre_oficial"])
dim_localidad["nombre_norm"] = dim_localidad["nombre_oficial"].apply(normalizar)

assert dim_localidad.shape[0] == 20, f"Se esperaban 20 localidades, hay {dim_localidad.shape[0]}"
assert dim_localidad["codigo_localidad"].nunique() == 20
print(dim_localidad)

    codigo_localidad      nombre_oficial         nombre_norm
0                  1             Usaquén             USAQUEN
1                  2           Chapinero           CHAPINERO
2                  3            Santa Fe            SANTA FE
3                  4       San Cristóbal       SAN CRISTOBAL
4                  5                Usme                USME
5                  6          Tunjuelito          TUNJUELITO
6                  7                Bosa                BOSA
7                  8             Kennedy             KENNEDY
8                  9            Fontibón            FONTIBON
9                 10            Engativá            ENGATIVA
10                11                Suba                SUBA
11                12      Barrios Unidos      BARRIOS UNIDOS
12                13         Teusaquillo         TEUSAQUILLO
13                14        Los Mártires        LOS MARTIRES
14                15      Antonio Nariño      ANTONIO NARINO
15                16    

In [10]:
# en_encuesta: True si el codigo_localidad aparece en la encuesta, False si no (Sumapaz)
codigos_en_encuesta = set(df["codigo_localidad"].unique())
dim_localidad["en_encuesta"] = dim_localidad["codigo_localidad"].isin(codigos_en_encuesta)

print("Localidades marcadas como NO encuestadas:")
print(dim_localidad.loc[~dim_localidad["en_encuesta"], ["codigo_localidad", "nombre_oficial"]])

assert (~dim_localidad["en_encuesta"]).sum() == 1, "Se esperaba exactamente 1 localidad fuera de la encuesta (Sumapaz)"
assert dim_localidad.loc[~dim_localidad["en_encuesta"], "nombre_oficial"].iloc[0] == "Sumapaz", \
    "La localidad marcada como no encuestada NO es Sumapaz — revisar codificación"

# sector_upl: moda de SectorUPL por codigo_localidad, tal como aparece en la encuesta
sector_por_localidad = (
    df.groupby("codigo_localidad")["SectorUPL"]
    .agg(lambda s: s.mode().iat[0] if not s.mode().empty else np.nan)
    .rename("sector_upl")
)

dim_localidad = dim_localidad.merge(sector_por_localidad, on="codigo_localidad", how="left")

print("\ndim_localidad con sector_upl (Sumapaz queda NaN, no está en la encuesta):")
print(dim_localidad)

Localidades marcadas como NO encuestadas:
    codigo_localidad nombre_oficial
19                20        Sumapaz

dim_localidad con sector_upl (Sumapaz queda NaN, no está en la encuesta):
    codigo_localidad      nombre_oficial         nombre_norm  en_encuesta  \
0                  1             Usaquén             USAQUEN         True   
1                  2           Chapinero           CHAPINERO         True   
2                  3            Santa Fe            SANTA FE         True   
3                  4       San Cristóbal       SAN CRISTOBAL         True   
4                  5                Usme                USME         True   
5                  6          Tunjuelito          TUNJUELITO         True   
6                  7                Bosa                BOSA         True   
7                  8             Kennedy             KENNEDY         True   
8                  9            Fontibón            FONTIBON         True   
9                 10            Engativá 

In [11]:
# Verificación empírica: ¿codigo_localidad de la encuesta coincide con dim_localidad?
mapeo_real_encuesta = (
    df[["codigo_localidad", "Localidad"]]
    .drop_duplicates()
    .sort_values("codigo_localidad")
)
mapeo_real_encuesta["Localidad_norm"] = mapeo_real_encuesta["Localidad"].apply(normalizar)

print("Mapeo real codigo_localidad -> Localidad, según la propia encuesta:")
print(mapeo_real_encuesta.to_string(index=False))

# Comparar contra dim_localidad (nuestra codificación DANE asumida)
comparacion = mapeo_real_encuesta.merge(
    dim_localidad[["codigo_localidad", "nombre_norm"]],
    on="codigo_localidad",
    how="left",
    suffixes=("_encuesta", "_dim")
)
comparacion["coincide"] = comparacion["Localidad_norm"] == comparacion["nombre_norm"]

print("\nComparación código por código:")
print(comparacion[["codigo_localidad", "Localidad", "nombre_norm", "coincide"]].to_string(index=False))

discrepancias = comparacion.loc[~comparacion["coincide"]]
if not discrepancias.empty:
    print(f"\nDISCREPANCIAS ENCONTRADAS ({len(discrepancias)} códigos no coinciden):")
    print(discrepancias[["codigo_localidad", "Localidad", "nombre_norm"]].to_string(index=False))
else:
    print("\nTodos los codigo_localidad de la encuesta coinciden con dim_localidad (codificación DANE confirmada).")

assert discrepancias.empty, "codigo_localidad de la encuesta NO sigue el orden asumido en dim_localidad — hay que reconstruir dim_localidad con el mapeo real de la encuesta, no con la lista DANE supuesta."

Mapeo real codigo_localidad -> Localidad, según la propia encuesta:
 codigo_localidad          Localidad     Localidad_norm
                1            Usaquén            USAQUEN
                2          Chapinero          CHAPINERO
                3           Santa Fe           SANTA FE
                4      San Cristóbal      SAN CRISTOBAL
                5               Usme               USME
                6         Tunjuelito         TUNJUELITO
                7               Bosa               BOSA
                8            Kennedy            KENNEDY
                9           Fontibón           FONTIBON
               10           Engativá           ENGATIVA
               11               Suba               SUBA
               12     Barrios Unidos     BARRIOS UNIDOS
               13        Teusaquillo        TEUSAQUILLO
               14       Los Mártires       LOS MARTIRES
               15     Antonio Nariño     ANTONIO NARINO
               16      Puente Aranda

In [12]:
RUTA_RIESGO = "../../outputs/riesgofeminicidio.csv"  # ajustar ruta relativa según ubicación del notebook

df_riesgo = pd.read_csv(RUTA_RIESGO, encoding="utf-8-sig", low_memory=False)
df_riesgo["Fecha"] = pd.to_datetime(df_riesgo["Fecha"])

print(f"Dimensiones: {df_riesgo.shape}  (esperado: (80, 6))")
print(f"Cortes disponibles: {sorted(df_riesgo['Fecha'].unique())}")

corte_reciente = df_riesgo["Fecha"].max()
print(f"\nCorte más reciente seleccionado: {corte_reciente.date()}")

df_riesgo_ultimo = df_riesgo.loc[df_riesgo["Fecha"] == corte_reciente].copy()
assert df_riesgo_ultimo.shape[0] == 20, f"Se esperaban 20 filas en el corte más reciente, hay {df_riesgo_ultimo.shape[0]}"

df_riesgo_ultimo["nombre_norm"] = df_riesgo_ultimo["Localidad"].apply(normalizar)

# Merge contra dim_localidad por nombre normalizado
dim_localidad = dim_localidad.merge(
    df_riesgo_ultimo[["nombre_norm", "PobMujeres"]],
    on="nombre_norm",
    how="left"
)

print("\ndim_localidad con PobMujeres incorporada:")
print(dim_localidad)

# Verificación: ninguna PobMujeres debe quedar nula (riesgofeminicidio.csv cubre las 20, incluida Sumapaz)
faltantes_pob = dim_localidad.loc[dim_localidad["PobMujeres"].isna(), ["codigo_localidad", "nombre_oficial"]]
if not faltantes_pob.empty:
    print("\nLocalidades SIN PobMujeres tras el merge (revisar nombre_norm):")
    print(faltantes_pob)
assert faltantes_pob.empty, "Hay localidades sin PobMujeres — revisar normalización de nombres"

print("\nMerge de PobMujeres: OK, 0 faltantes.")

Dimensiones: (80, 6)  (esperado: (80, 6))
Cortes disponibles: [Timestamp('2025-06-30 00:00:00'), Timestamp('2025-09-30 00:00:00'), Timestamp('2025-12-31 00:00:00'), Timestamp('2026-03-31 00:00:00')]

Corte más reciente seleccionado: 2026-03-31

dim_localidad con PobMujeres incorporada:
    codigo_localidad      nombre_oficial         nombre_norm  en_encuesta  \
0                  1             Usaquén             USAQUEN         True   
1                  2           Chapinero           CHAPINERO         True   
2                  3            Santa Fe            SANTA FE         True   
3                  4       San Cristóbal       SAN CRISTOBAL         True   
4                  5                Usme                USME         True   
5                  6          Tunjuelito          TUNJUELITO         True   
6                  7                Bosa                BOSA         True   
7                  8             Kennedy             KENNEDY         True   
8                  9

In [13]:
def verificar_cruce(nombre_fuente, df_fuente, col_localidad, tolera_sumapaz=False, tolera_solo_codigo=False, col_codigo=None):
    """
    Verifica que todas las filas de df_fuente emparejen contra dim_localidad.
    - Si col_codigo se pasa, se compara por código (fuentes tipo llamadas 123).
    - Si no, se compara por nombre normalizado.
    tolera_sumapaz=True permite que Sumapaz quede sin match (fuentes de encuesta).
    """
    print(f"\n{'='*70}\n{nombre_fuente}\n{'='*70}")

    if col_codigo:
        codigos_fuente = set(df_fuente[col_codigo].dropna().unique())
        codigos_dim = set(dim_localidad["codigo_localidad"].unique())
        no_match = codigos_fuente - codigos_dim
    else:
        df_fuente = df_fuente.copy()
        df_fuente["_nombre_norm"] = df_fuente[col_localidad].apply(normalizar)
        nombres_fuente = set(df_fuente["_nombre_norm"].dropna().unique())
        nombres_dim = set(dim_localidad["nombre_norm"].unique())
        no_match = nombres_fuente - nombres_dim

    if no_match:
        print(f"Valores en '{nombre_fuente}' SIN match en dim_localidad: {no_match}")
    else:
        print(f"Todos los valores de '{nombre_fuente}' emparejan correctamente.")

    # Chequeo inverso: localidades de dim_localidad ausentes en la fuente
    if col_codigo:
        ausentes = codigos_dim - codigos_fuente
        ausentes_nombres = dim_localidad.loc[dim_localidad["codigo_localidad"].isin(ausentes), "nombre_oficial"].tolist()
    else:
        ausentes = nombres_dim - nombres_fuente
        ausentes_nombres = dim_localidad.loc[dim_localidad["nombre_norm"].isin(ausentes), "nombre_oficial"].tolist()

    if tolera_sumapaz:
        ausentes_no_tolerados = [n for n in ausentes_nombres if n != "Sumapaz"]
    else:
        ausentes_no_tolerados = ausentes_nombres

    print(f"Localidades de dim_localidad ausentes en '{nombre_fuente}': {ausentes_nombres}")
    if ausentes_no_tolerados:
        print(f"NO TOLERADO: {ausentes_no_tolerados}")
    else:
        print("(sin ausencias no toleradas)" if not tolera_sumapaz else "Solo Sumapaz ausente — tolerado.")

    return no_match, ausentes_no_tolerados


resultados_verificacion = {}

# --- Fuentes administrativas: 0 no-coincidencias toleradas ---
resultados_verificacion["riesgofeminicidio"] = verificar_cruce(
    "riesgofeminicidio.csv", df_riesgo, col_localidad="Localidad", tolera_sumapaz=False
)

RUTA_DUPLAS = "../../outputs/duplas.csv"
df_duplas = pd.read_csv(RUTA_DUPLAS, encoding="utf-8-sig", low_memory=False)
resultados_verificacion["duplas"] = verificar_cruce(
    "duplas.csv", df_duplas, col_localidad="Localidad", tolera_sumapaz=False
)

RUTA_LP = "../../outputs/lineapurpura.csv"
df_lp = pd.read_csv(RUTA_LP, encoding="utf-8-sig", low_memory=False)
resultados_verificacion["lineapurpura"] = verificar_cruce(
    "lineapurpura.csv", df_lp, col_localidad="Localidad", tolera_sumapaz=False
)

RUTA_DELITOS = "../../outputs/delitossexuales.csv"
df_delitos = pd.read_csv(RUTA_DELITOS, encoding="utf-8-sig", low_memory=False)
resultados_verificacion["delitossexuales"] = verificar_cruce(
    "delitossexuales.csv", df_delitos, col_localidad="Localidad", tolera_sumapaz=False
)

RUTA_LLAMADAS = "../../outputs/llamadas123_consolidado_limpio.csv"

df_llamadas = pd.read_csv(
    RUTA_LLAMADAS,
    encoding="utf-8-sig",
    sep=";",                # <- el archivo usa punto y coma, no coma
    low_memory=False
)

print(f"Dimensiones: {df_llamadas.shape}")
print(df_llamadas.columns.tolist())
print(df_llamadas.head(3))

# CODIGO_LOCALIDAD debe quedar numérica
df_llamadas["CODIGO_LOCALIDAD"] = pd.to_numeric(df_llamadas["CODIGO_LOCALIDAD"], errors="coerce")
print(f"\nValores nulos en CODIGO_LOCALIDAD tras conversión: {df_llamadas['CODIGO_LOCALIDAD'].isna().sum()}")
print(f"Códigos únicos: {sorted(df_llamadas['CODIGO_LOCALIDAD'].dropna().unique())}")

resultados_verificacion["llamadas123"] = verificar_cruce(
    "llamadas123_consolidado_limpio.csv", df_llamadas,
    col_localidad=None, col_codigo="CODIGO_LOCALIDAD", tolera_sumapaz=False
)

# --- Fuente de encuesta: Sumapaz tolerado ---
resultados_verificacion["encuesta_percepcion"] = verificar_cruce(
    "Encuesta_percepcion (df)", df, col_localidad=None, col_codigo="codigo_localidad", tolera_sumapaz=True
)

# --- Resumen final ---
print(f"\n{'='*70}\nRESUMEN\n{'='*70}")
hubo_error = False
for fuente, (no_match, ausentes_no_tolerados) in resultados_verificacion.items():
    estado = "OK" if not no_match and not ausentes_no_tolerados else "FALLA"
    if estado == "FALLA":
        hubo_error = True
    print(f"{fuente:30} -> {estado}")

assert not hubo_error, "Hay fuentes con no-coincidencias no toleradas — revisar arriba antes de continuar."
print("\nVerificación cruzada completa: todas las fuentes administrativas emparejan al 100%, Sumapaz tolerado solo en encuesta.")


riesgofeminicidio.csv
Todos los valores de 'riesgofeminicidio.csv' emparejan correctamente.
Localidades de dim_localidad ausentes en 'riesgofeminicidio.csv': []
(sin ausencias no toleradas)

duplas.csv
Todos los valores de 'duplas.csv' emparejan correctamente.
Localidades de dim_localidad ausentes en 'duplas.csv': []
(sin ausencias no toleradas)

lineapurpura.csv
Todos los valores de 'lineapurpura.csv' emparejan correctamente.
Localidades de dim_localidad ausentes en 'lineapurpura.csv': []
(sin ausencias no toleradas)

delitossexuales.csv
Todos los valores de 'delitossexuales.csv' emparejan correctamente.
Localidades de dim_localidad ausentes en 'delitossexuales.csv': []
(sin ausencias no toleradas)
Dimensiones: (52717, 13)
['NUMERO_INCIDENTE', 'FECHA_INICIO_DESPLAZAMIENTO_MOVIL', 'CODIGO_LOCALIDAD', 'LOCALIDAD', 'EDAD', 'UNIDAD', 'GENERO', 'TIPO_INCIDENTE', 'PRIORIDAD_FINAL', 'RECEPCION', 'ARCHIVO_ORIGEN', 'MES', 'AÑO']
  NUMERO_INCIDENTE FECHA_INICIO_DESPLAZAMIENTO_MOVIL  CODIGO_LOC

#Celda — Cierre del Paso 1.5: constancia en docs/supuestos.md

In [14]:
n_nulos_llamadas = df_llamadas["CODIGO_LOCALIDAD"].isna().sum()
pct_nulos_llamadas = n_nulos_llamadas / len(df_llamadas) * 100

texto_supuesto_1_5 = f"""
## Paso 1.5 — Construcción de `dim_localidad` (registrado {pd.Timestamp.now().strftime('%Y-%m-%d')})

Se construyó `dim_localidad` con 20 filas (`codigo_localidad` 1–20, `nombre_oficial`,
`nombre_norm`, `sector_upl`, `en_encuesta`, `PobMujeres`), usando `normalizar()` =
`unidecode().upper().strip()` con espacios múltiples colapsados.

**Codificación confirmada empíricamente:** se cruzó `codigo_localidad` de la Encuesta de
Percepción contra su propia columna `Localidad` (texto) y coincide exactamente con la
codificación DANE estándar asumida (1=Usaquén ... 19=Ciudad Bolívar, 20=Sumapaz). No se
asumió a ciegas — se verificó código por código.

**`en_encuesta`:** única localidad marcada `False` es Sumapaz (código 20), como se esperaba.

**`PobMujeres`:** extraída de `riesgofeminicidio.csv`, corte más reciente ({corte_reciente.date()}),
sin faltantes en las 20 localidades.

**Verificación cruzada de las 6 fuentes contra `dim_localidad`:**

| Fuente | Resultado |
|---|---|
| `riesgofeminicidio.csv` | OK — 0 no coincidencias |
| `duplas.csv` | OK — 0 no coincidencias |
| `lineapurpura.csv` | OK — 0 no coincidencias |
| `delitossexuales.csv` | OK — 0 no coincidencias |
| `llamadas123_consolidado_limpio.csv` | OK — 0 no coincidencias (tras corregir separador `;` y encoding) |
| Encuesta de Percepción (`df`) | OK — solo Sumapaz ausente, tolerado por diseño |

**Excepción documentada — `llamadas123_consolidado_limpio.csv`:** {n_nulos_llamadas} registros
({pct_nulos_llamadas:.2f}% de {len(df_llamadas)}) tienen `CODIGO_LOCALIDAD` nulo tras la
conversión numérica. Se excluyen de toda agregación territorial (Nivel 2 de la jerarquía
de integración) por no tener llave de cruce válida contra `dim_localidad`. No se imputan.
Su magnitud (<0,5%) se considera marginal frente al volumen total del dataset.

**Nota técnica:** `llamadas123_consolidado_limpio.csv` usa separador `;` (no `,`) y
codificación con BOM (`utf-8-sig`) — debe cargarse con `sep=";"` explícito o las 13
columnas colapsan en una sola.
"""

with open("../../docs/supuestos.md", "a", encoding="utf-8") as f:
    f.write(texto_supuesto_1_5)

print("Constancia del Paso 1.5 agregada a docs/supuestos.md")
print(f"\ndim_localidad final ({dim_localidad.shape[0]} filas):")
print(dim_localidad)

Constancia del Paso 1.5 agregada a docs/supuestos.md

dim_localidad final (20 filas):
    codigo_localidad      nombre_oficial         nombre_norm  en_encuesta  \
0                  1             Usaquén             USAQUEN         True   
1                  2           Chapinero           CHAPINERO         True   
2                  3            Santa Fe            SANTA FE         True   
3                  4       San Cristóbal       SAN CRISTOBAL         True   
4                  5                Usme                USME         True   
5                  6          Tunjuelito          TUNJUELITO         True   
6                  7                Bosa                BOSA         True   
7                  8             Kennedy             KENNEDY         True   
8                  9            Fontibón            FONTIBON         True   
9                 10            Engativá            ENGATIVA         True   
10                11                Suba                SUBA       